# Chotu X-ray 01 · Inside the tokenizer, the data and the untrained model

Every picture here comes from **your real objects**: the SmolLM2 tokenizer, your token shards, and the actual 180.9M-parameter Chotu model. Nothing is a mock-up.

**How to learn with it:** run a cell, look, then answer the question under it *before* reading on. Stuck? Ask Claude: *"Teach me Socratically: one question at a time, wait for my answer, use a diagram or analogy, no maths. Topic: …"*. Write your answers in `docs/LEARNING.md`.

The model here is **untrained** (random weights). Keep these pictures: after Session 3 you'll run the same cells on a trained checkpoint and compare.

In [ ]:
# 0. Setup
%matplotlib inline
%pip install -q matplotlib==3.11.2
import glob, sys
from pathlib import Path
from IPython.display import HTML, display
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
STORAGE = Path.home() / 'chotu-data'   # same folder the Session 2 notebook writes to
from transformers import AutoTokenizer
from chotu.config import TOKENIZER_ID, chotu_config
from chotu import xray as X
import importlib; importlib.reload(X)
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID)
print('repo:', REPO, '| data:', STORAGE)

## 1 · Tokenizer lens: what the model actually reads

The model never sees letters. It sees a list of numbers, one per *token* (a common chunk of text). Each coloured box is one token; the number under it is its id, which is all the model gets.

In [ ]:
for text in [
    "The cat sat on the mat.",
    "In 2048, Chotu learned that photosynthesis isn't magic.",
    "antidisestablishmentarianism",
    "  def add(a, b):\n      return a + b",
    "Namaste! मेरा नाम चोटू है",
]:
    display(HTML(X.token_lens(tok, text)))

In [ ]:
# Try your own sentence
display(HTML(X.token_lens(tok, "Type anything here and run the cell again")))

**Think about it**
1. Why does "2048" become four tokens while "cat" is one? What might that do for arithmetic?
2. Why does the Hindi sentence cost so many more tokens than English? What does that mean for a model trained mostly on English?
3. Notice `␣` (a space) is usually glued to the *front* of a word. Why might that be a good design?

## 2 · Shard microscope: the real training data

This is a random window of your actual shard file: raw ids, then what they decode to. The red marker is the end-of-document token (id 0) that the data script puts between documents.

In [ ]:
shards = sorted(glob.glob(str(STORAGE / 'shards' / 'train_*.bin'))) or sorted(glob.glob(str(STORAGE / 'test' / 'shards' / 'train_*.bin')))
assert shards, 'No shards yet: run cell 3 (quick test) of session2_data_local.ipynb first'
print(len(shards), 'shard file(s), e.g.', shards[0])
for seed in [1, 2, 3]:
    display(HTML(X.shard_window(tok, shards[0], n=120, seed=seed)))

In [ ]:
counts, total = X.token_frequency(shards, max_tokens=50_000_000)
fig = X.plot_token_frequency(tok, counts, total)

**Think about it**
1. The top 10 tokens cover about a quarter of all text. If the model only learned those, how good would its guesses be, and what would it still get wrong?
2. Most of the 49,152 ids are rare. How many times might the model see a rare token in 10B tokens? Is that enough to learn it?
3. Why does the model need the red end-of-document marker at all?

## 3 · Parameter map: where the 180.9M numbers live

In [ ]:
import torch
from transformers import Qwen3ForCausalLM
torch.manual_seed(0)
model = Qwen3ForCausalLM(chotu_config(attn_implementation='eager'))  # eager = lets us read attention
rows, total = X.param_breakdown(model)
for k, v in rows.items(): print(f'{k:42s} {v/1e6:7.2f}M  {v/total:5.1%}')
print(f'{"total":42s} {total/1e6:7.2f}M')
fig = X.plot_param_map(model)

**Think about it**
1. Most parameters are in the MLPs, not attention. If attention decides *where to look*, what might the MLPs be storing?
2. k_proj and v_proj are half the size of q_proj. That's grouped-query attention (5 key/value heads shared by 10 query heads). What does sharing save when the model is generating text on your laptop?
3. The word table is used twice (reading words in, writing words out). What's the advantage at our small size?

## 4 · Forward-pass X-ray: one sentence through the untrained model

We send one sentence through all 32 layers and record what happens inside. **Prediction first:** with random weights, what do you expect to see?

In [ ]:
x = X.xray(model, tok, "The capital of France is Paris, and the capital of Italy is")
print('Top-10 next-token guesses (untrained):')
for t, p in x['next']: print(f'  {t!r:18s} {p:.5%}')
print(f'Pure chance would be 1/49,152 = {1/49152:.5%}')

In [ ]:
fig = X.plot_attention(x, layer=0)
fig = X.plot_attention(x, layer=31)

In [ ]:
fig = X.plot_logit_lens(x, every=2)

In [ ]:
fig = X.plot_layer_signal(x)

**Think about it**
1. The attention maps are triangles: nothing above the diagonal. Why can't a word look at words that come *after* it?
2. Many heads look hard at the first word. With random weights, why might the first position attract attention?
3. In the logit lens, the embedding row confidently predicts the *current* word as the next word. Why does sharing the word table cause that, and why will training fix it?
4. After training, what should the logit lens look like for "the capital of Italy is"? At roughly which layers would you expect "Rome" to appear?

**Save these pictures.** After Session 3 we'll load a trained checkpoint into the same cells. The difference between the two runs is what training actually did.